In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import inv, matrix_rank, det

print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)

NumPy  : 2.4.2
Pandas : 2.3.3


In [12]:
import numpy as np
from numpy.linalg import inv, matrix_rank


def det_checker(X):
    m, d = X.shape
    if m == d:
        return "even"
    elif m > d:
        return "over"
    else:
        return "under"


def RC_checker(X, y):
    X_aug = np.append(X, y, axis=1)
    rankX = matrix_rank(X)
    rankX_ = matrix_rank(X_aug)
    d = X.shape[1]
    if rankX == rankX_:
        RC = 1 if rankX == d else 3
    else:
        RC = 2
    return RC, rankX, rankX_


def evenSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X) @ y, "Unique solution."
    elif RC == 2:
        return None, "No solution."
    else:
        return None, "Infinitely many solutions."


def overSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 1:
        return inv(X.T @ X) @ X.T @ y, "Unique solution."
    elif RC == 3:
        return None, "Infinitely many solutions."
    else:
        return inv(X.T @ X) @ X.T @ y, "No exact sol, least square approx."


def underSolver(X, y):
    RC, _, _ = RC_checker(X, y)
    if RC == 2:
        return None, "No solution."
    else:
        return X.T @ inv(X @ X.T) @ y, "No exact sol, least norm approx."


def solveLE(X, y):
    det = det_checker(X)
    if det == "even":
        w, ans = evenSolver(X, y)
    elif det == "over":
        w, ans = overSolver(X, y)
    else:
        w, ans = underSolver(X, y)
    print(ans, "\nw =", w)
    return w


# --- Quick diagnostic ---
X = np.array([[1, 3], [1, 4], [1, 5], [1, 6], [1, 7]])
y = np.array([[5], [4], [3], [2], [1]])

print("System type:", det_checker(X))
RC, rX, rX_ = RC_checker(X, y)
print(f"rank(X)={rX}, rank(X~)={rX_}, case={RC}")
w = solveLE(X, y)

System type: over
rank(X)=2, rank(X~)=2, case=1
Unique solution. 
w = [[ 8.]
 [-1.]]


In [13]:
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import inv
import numpy as np


def polyTx(X, order):
    """Polynomial feature matrix with bias column. Shape: (N, order+1)."""
    return PolynomialFeatures(order).fit_transform(X)


def solvePR(P, y, ridge=False, lamb=0.01):
    """Solve polynomial regression. Auto primal (N>M) or dual (N<M)."""
    if ridge:
        if P.shape[0] > P.shape[1]:  # Primal
            w = inv(P.T @ P + lamb * np.eye(P.shape[1])) @ P.T @ y
        else:  # Dual
            w = P.T @ inv(P @ P.T + lamb * np.eye(P.shape[0])) @ y
    else:
        if P.shape[0] > P.shape[1]:  # Primal
            w = inv(P.T @ P) @ P.T @ y
        else:  # Dual
            w = P.T @ inv(P @ P.T) @ y
    return w


def solveLE_Ridge(X, y, lamb=0.01):
    """Linear regression with Ridge. X must already include bias column."""
    return solvePR(X, y, ridge=True, lamb=lamb)


# Quick test
X_test_poly = np.array([[-10], [-8], [-3], [-1], [2], [8]])
y_test_poly = np.array([[5], [5], [4], [3], [2], [2]])
P = polyTx(X_test_poly, 3)
print("P shape:", P.shape)  # (6, 4): [1, x, x^2, x^3]
w = solvePR(P, y_test_poly)
print("w:", w.ravel())

P shape: (6, 4)
w: [ 2.68935636 -0.37722517  0.01343815  0.00285772]


In [14]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([3, 4, 5, 6, 7]).reshape(-1, 1)
y = np.array([5, 4, 3, 2, 1]).reshape(-1, 1)

# --- With bias ---
bias = np.ones((X.shape[0], 1))
X_b = np.hstack((bias, X))  # [1, x]
w = solveLE(X_b, y)  # w = [w0_bias, w1_slope]
# Predict for new inputs
X_test   = np.array([[8]])
X_test_b = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
y_pred   = X_test_b @ w
print('Predictions:', y_pred.ravel())

# # --- Without bias ---
# w_nb = solveLE(X, y)

# # --- Plot both ---
# plt.scatter(X, y, color='blue', label='Training Samples', marker='o')
# plt.plot(X, X_b @ w,    color='red',   label='With bias')
# plt.plot(X, X   @ w_nb, color='green', label='No bias')
# plt.xlabel('Students'); plt.ylabel('Books sold'); plt.legend(); plt.show()

Unique solution. 
w = [[ 8.]
 [-1.]]
Predictions: [4.4408921e-15]


In [15]:
X = np.array([1, 4, 2, 7, -3, 11]).reshape(3, 2)
y = np.array([1, -2.5, 4]).reshape(-1, 1)
print(X)
X_aug = np.append(X, y, axis=1)
print("\nrank(X)  =", matrix_rank(X))
print("rank(X~) =", matrix_rank(X_aug))

[[ 1  4]
 [ 2  7]
 [-3 11]]

rank(X)  = 2
rank(X~) = 3


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([1, 2, 0, 6, 1, 0, 0, 5, 1, 7]).reshape(-1, 2)
y = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)

# --- With bias ---
bias = np.ones((X.shape[0], 1))
X_b = np.hstack((bias, X))  # [1, x]
w = solveLE(X_b, y)  # w = [w0_bias, w1_slope]
# Predict for new inputs
X_test = np.array([[1, 3]])
X_test_b = np.hstack((np.ones((X_test.shape[0], 1)), X_test))
y_pred = X_test_b @ w
print("Predictions:", y_pred.ravel())

y_full = X_b @ w
print("Predictions:", y_pred.ravel())

# # --- Without bias ---
# w_nb = solveLE(X, y)

# # --- Plot both ---
# plt.scatter(X, y, color='blue', label='Training Samples', marker='o')
# plt.plot(X, X_b @ w,    color='red',   label='With bias')
# plt.plot(X, X   @ w_nb, color='green', label='No bias')
# plt.xlabel('Students'); plt.ylabel('Books sold'); plt.legend(); plt.show()
y_full

No exact sol, least square approx. 
w = [[1.13207547]
 [0.8490566 ]
 [0.33962264]]
Predictions: [3.]
Predictions: [3.]


array([[2.66037736],
       [3.16981132],
       [1.98113208],
       [2.83018868],
       [4.35849057]])

: 

In [20]:
from sklearn.metrics import mean_squared_error
MSE = mean_squared_error(y_full, y)
MSE

1.388679245283019